# Tutorial 1: OneEHR Quickstart

This tutorial demonstrates the complete OneEHR workflow using the bundled TJH COVID-19 dataset.

## Install

```bash
pip install oneehr
```

In [ ]:
import oneehr
print(f"OneEHR version: {oneehr.__version__}")

## Step 1: Define the Experiment Config

OneEHR uses a single TOML file to define the entire experiment.

In [ ]:
config_text = """
[dataset]
dynamic = "examples/tjh/dynamic.csv"
static = "examples/tjh/static.csv"
label = "examples/tjh/label_mortality.csv"

[preprocess]
bin_size = "1d"
top_k_codes = 50
pipeline = [
  { op = "forward_fill", cols = "num__*", group_key = "patient_id", order_key = "bin_time" },
  { op = "standardize", cols = "num__*" },
]

[task]
kind = "binary"
prediction_mode = "patient"

[split]
kind = "random"
seed = 42
val_size = 0.15
test_size = 0.15

[[models]]
name = "xgboost"
[models.params]
max_depth = 4
n_estimators = 100

[[models]]
name = "gru"

[trainer]
device = "cpu"
seed = 42
max_epochs = 20
batch_size = 32
lr = 0.001

[output]
root = "runs"
run_name = "tutorial_quickstart"
"""

# Save config
from pathlib import Path
cfg_path = Path("tutorial_config.toml")
cfg_path.write_text(config_text)
print("Config saved.")

## Step 2: Preprocess

Bin events into time windows, build feature matrices, and split patients.

In [ ]:
config = oneehr.load_config(str(cfg_path))
result = oneehr.preprocess(config)
print(f"Preprocessed: {result}")

## Step 3: Train

Train all models defined in the config (XGBoost + GRU).

In [ ]:
train_result = oneehr.train(config)
print(f"Training complete: {train_result}")

## Step 4: Test

Evaluate all trained models on the held-out test set.

In [ ]:
test_result = oneehr.test(config)
print(f"Test complete: {test_result}")

## Step 5: Analyze

Run cross-system comparison with bootstrap CI and statistical tests.

In [ ]:
analyze_result = oneehr.analyze(config)
print(f"Analysis complete: {analyze_result}")

## Step 6: Visualize Results

Generate publication-quality figures.

In [ ]:
import json
import pandas as pd

run_dir = config.run_dir()

# Load metrics
metrics = json.loads((run_dir / "test" / "metrics.json").read_text())
print("\nTest Metrics:")
for system, m in metrics.items():
    if isinstance(m, dict):
        auroc = m.get("auroc", "N/A")
        auprc = m.get("auprc", "N/A")
        print(f"  {system}: AUROC={auroc:.4f}, AUPRC={auprc:.4f}")

## Summary

In this tutorial you:
1. Defined an experiment with a single TOML config
2. Preprocessed EHR events into binned features
3. Trained XGBoost (ML) and GRU (DL) models
4. Evaluated both on a held-out test set
5. Ran cross-system comparison with statistical tests

All artifacts are stored in `runs/tutorial_quickstart/` for reproducibility.